# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/DanialHameed/fly/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

In [1]:
import os
from pathlib import Path

here = Path.cwd()
for candidate in [here, *here.parents]:
    if (candidate / "data" / "raw" / "content_refresh_anonymized.csv").exists():
        os.chdir(candidate)
        break
else:
    if not os.path.exists("/content/fly"):
        get_ipython().system('git clone https://github.com/DanialHameed/fly.git /content/fly')
    os.chdir("/content/fly")

print("Working directory:", os.getcwd())

import pandas as pd
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)

# Grounding the methodology questions in the paper's own reported numbers,
# quoted here so the discussion below stays tied to specific, checkable facts
# rather than a vague impression of the PDF.
paper_facts = {
    "health_score_formula": "Impressions (30 pts) + Position (30 pts) + CTR (20 pts) + Scroll depth (20 pts)",
    "rf_feature_importance_top3": {"avg_position": 0.43, "impressions": 0.32, "scroll_depth": 0.15},
    "rf_predicting": "health_score",
    "ml_pipeline_split": "Random Forest (80/20 split), Logistic Regression (80/20 split), Decision Tree (80/20 split)",
    "ml_pipeline_n_brands": 57,
    "ml_pipeline_n_content": "61.8K (active-content feature-vector subset)",
    "logistic_regression_holdout_accuracy": 0.71,
    "paper_self_disclosed_caveat": "the target itself is partly constructed from some of these inputs, so importance is descriptive rather than causal",
}
for k, v in paper_facts.items():
    print(f"{k}: {v}")

Working directory: C:\Users\Laptop\Documents\fly


health_score_formula: Impressions (30 pts) + Position (30 pts) + CTR (20 pts) + Scroll depth (20 pts)
rf_feature_importance_top3: {'avg_position': 0.43, 'impressions': 0.32, 'scroll_depth': 0.15}
rf_predicting: health_score
ml_pipeline_split: Random Forest (80/20 split), Logistic Regression (80/20 split), Decision Tree (80/20 split)
ml_pipeline_n_brands: 57
ml_pipeline_n_content: 61.8K (active-content feature-vector subset)
logistic_regression_holdout_accuracy: 0.71
paper_self_disclosed_caveat: the target itself is partly constructed from some of these inputs, so importance is descriptive rather than causal


### 1. Two paper findings + my methodology questions

**Finding: ML Appendix — "What Predicts Health?"** (Random Forest, `avg_position` 43% importance,
`impressions` 32%, `scroll_depth` 15%). **Methodology question:** the paper's own Health Score
formula is `Impressions (30 pts) + Position (30 pts) + CTR (20 pts) + Scroll depth (20 pts)` — and
the top three "predictors" of that score are three of its own four ingredients. The paper already
discloses this ("the target itself is partly constructed from some of these inputs, so importance
is descriptive rather than causal"), which is exactly the right instinct — but I'd ask one
question further: **would the same features still dominate importance against a target that isn't
partly built from them** — say, raw future impressions 30 days out, or an independent human
quality rating? That test would separate "the model is recovering its own formula" (which this
result is consistent with) from "these signals have real predictive power on an outcome the model
wasn't shown." Right now the finding can't tell those two apart, and the paper's own caveat says
as much — I'm asking for the follow-up experiment that would resolve it, not disputing the
disclosure.

**Finding: ML Appendix — "What Predicts Growth?"** (Logistic Regression, 71% holdout accuracy,
`Content Age`, `Days Since Update`, `Days Visible` as the strongest signals). **Methodology
question:** the paper's own Methodology section states the ML pipeline uses an 80/20 split for
each model, with no mention of grouping by brand. The dataset spans 57 brands and 61.8K content
pieces — meaning, on average, over 1,000 pages per brand, almost certainly sharing brand-level
patterns (a redesign, a seasonal campaign, an editorial policy) the same way client pages did in
our own starter dataset. **Would a brand-grouped holdout (no brand's pages in both train and
test) still report 71% accuracy, or would it drop** the way our own model's precision@50 drops
from 0.90 to 0.64 once we move from a random split to a client-grouped one (section 2 below)? I'm
not claiming the paper's number is wrong — I'm asking whether it's been stress-tested against the
specific leakage path our own Week 5/6 work found to matter a lot on a structurally similar
dataset.

## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

In [2]:
import numpy as np
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

SEED = 42
df["has_word_count"] = df["word_count"].notna().astype(int)
df["has_keyword_data"] = df["search_volume"].notna().astype(int)
df["has_position"] = (df["avg_position"] > 0).astype(int)
df["log_impressions_90d"] = np.log1p(df["impressions_90d"])
df["log_clicks_90d"] = np.log1p(df["clicks_90d"])
df["log_sessions_90d"] = np.log1p(df["sessions_90d"])
df["log_ai_sessions_90d"] = np.log1p(df["ai_sessions_90d"])

NUMERIC = ["log_impressions_90d", "log_clicks_90d", "log_sessions_90d", "log_ai_sessions_90d",
    "days_with_impressions", "days_with_sessions", "content_age_days", "days_since_last_update",
    "ctr", "avg_position", "engagement_rate", "scroll_rate", "ai_traffic_pct", "word_count",
    "char_count", "search_volume", "competition", "cpc", "has_word_count", "has_keyword_data", "has_position"]
CATEGORICAL = ["content_type", "main_intent", "competition_level", "age_tier"]
FEATURES = NUMERIC + CATEGORICAL

X = df[FEATURES]
y = df["is_declining_label"]
groups = df["client_id"]

pre = ColumnTransformer([
    ("num", Pipeline([("impute", SimpleImputer(strategy="constant", fill_value=0)), ("scale", StandardScaler())]), NUMERIC),
    ("cat", Pipeline([("impute", SimpleImputer(strategy="constant", fill_value="unknown")),
                      ("onehot", OneHotEncoder(handle_unknown="ignore"))]), CATEGORICAL),
])

def precision_at_k(y_true, scores, k):
    order = np.argsort(-np.asarray(scores))
    return float(np.asarray(y_true)[order[:k]].mean())

def run_split(train_idx, test_idx, label):
    Xtr, Xte = X.iloc[train_idx], X.iloc[test_idx]
    ytr, yte = y.iloc[train_idx], y.iloc[test_idx]
    pipe = Pipeline([("pre", pre), ("clf", RandomForestClassifier(
        n_estimators=300, max_depth=8, min_samples_leaf=20, random_state=SEED, n_jobs=1))])
    pipe.fit(Xtr, ytr)
    proba = pipe.predict_proba(Xte)[:, 1]
    train_clients = set(df.iloc[train_idx]["client_id"])
    test_clients = set(df.iloc[test_idx]["client_id"])
    row = {"split": label, "client_overlap": len(train_clients & test_clients), "n_test": len(test_idx)}
    for k in [10, 20, 50, 100]:
        row[f"precision@{k}"] = round(precision_at_k(yte, proba, k), 3)
    return row

# BEFORE: naive random row split -- looks standard, but a client's other pages
# can land in both train and test, letting the model learn "this client's
# baseline" rather than generalizable signal.
tr_naive, te_naive = train_test_split(np.arange(len(df)), test_size=0.25, random_state=SEED, stratify=y)
row_naive = run_split(tr_naive, te_naive, "naive_random_split")

# AFTER: grouped client-holdout split, same as Week 5.
gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=SEED)
tr_grp, te_grp = next(gss.split(X, y, groups))
row_grp = run_split(tr_grp, te_grp, "grouped_client_holdout")

import pandas as pd
result = pd.DataFrame([row_naive, row_grp]).set_index("split")
print(result.to_string())

                        client_overlap  n_test  precision@10  precision@20  precision@50  precision@100
split                                                                                                  
naive_random_split                  31    7500           0.9           0.8          0.90           0.87
grouped_client_holdout               0    7115           0.5           0.4          0.64           0.65


### 2. My model under an honest split (before/after)

The gap is large and in the expected direction. **Naive random row split**: 31 of the same
clients appear in *both* train and test, and precision@50 comes out to **0.90** (precision@100:
0.87) — suspiciously close to perfect for a problem this genuinely hard. **Grouped client-holdout
split** (same model, same features, same seed, zero client overlap): precision@50 drops to
**0.64** (precision@100: 0.65). That's a 26-point inflation from a split choice alone, nothing
else changed. This is the concrete "before/after" the naive split hides: a model that looks
close to production-ready on a random split is actually learning "which client is this," not
"is this page declining," to a meaningful degree — and it's the direct evidence behind the
methodology question I raised above about the paper's own un-grouped 80/20 splits.

## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

In [3]:
# Same leakage guard as weeks 3-5: the label source and IDs must never be features.
leaky = set(FEATURES) & {"trend_direction", "trend_pct", "is_declining_label", "content_id", "client_id"}
print("Label-derived columns in FEATURES?", bool(leaky), "->", leaky if leaky else "none")

# Generalized smell test (inspired by the paper's own Health Score circularity,
# finding #1 in section 1 above): does any single feature correlate with the
# label suspiciously strongly, the way "avg_position" would with a target that
# is PARTLY DEFINED from avg_position?
corr = df[NUMERIC].corrwith(df["is_declining_label"]).sort_values(key=lambda s: s.abs(), ascending=False)
print("\nFeature correlations with is_declining_label, strongest first:")
print(corr.round(3).to_string())
FLAG = 0.9
suspicious = corr[corr.abs() > FLAG]
print(f"\nFeatures above |{FLAG}| (would need investigation): {list(suspicious.index) if len(suspicious) else 'none'}")
print(f"Max |correlation| observed: {corr.abs().max():.3f}")

Label-derived columns in FEATURES? False -> none

Feature correlations with is_declining_label, strongest first:
has_position              0.220
days_with_impressions     0.190
log_impressions_90d       0.177
content_age_days         -0.164
has_keyword_data          0.146
has_word_count            0.090
word_count                0.090
days_since_last_update    0.081
char_count                0.072
ctr                      -0.062
avg_position             -0.029
days_with_sessions       -0.025
search_volume            -0.019
cpc                      -0.017
log_sessions_90d          0.015
engagement_rate          -0.013
competition              -0.009
log_ai_sessions_90d      -0.004
log_clicks_90d            0.003
scroll_rate              -0.003
ai_traffic_pct            0.002

Features above |0.9| (would need investigation): none
Max |correlation| observed: 0.220


### 3. Leakage audit

No label-derived columns reached `FEATURES` (checked in code, not assumed). The correlation smell
test — the same kind of check the paper's Health Score problem calls for — comes back clean:
the strongest single-feature correlation with the label is `has_position` at 0.220, far below the
0.9 flag threshold that would suggest a feature is quietly restating the label the way
`avg_position` restates part of Health Score. This is a different, milder leakage risk than
Week 3's warehouse trap (a snapshot dimension leaking a future date) — the starter CSV is a
single pre-aggregated export per page, not a daily panel, so that specific trap doesn't apply
here architecturally. The leakage risk that *did* apply and *did* matter this week was the split
design audited in section 2, not a smuggled-in feature.

## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


### 4. Claim rewrite

**Original (Week 5, section 3):** "Every method beats the recomputed Week 4 baseline at every K
on this exact test split — that part is unsurprising and reassuring (**the leakage guard and
grouped split are working, not accidentally inflating everyone equally**)."

That parenthetical oversteps what the evidence at the time actually supported. "Every model beats
the baseline" is a comparative pattern; "the split is working" is a claim about validation design,
and at the time I had no direct evidence for it — I inferred it indirectly, which is exactly the
kind of gap this assignment is about closing.

**Rewritten, in safe language:** "Every method beats the recomputed Week 4 baseline at every K on
this test split — a comparative pattern, not by itself evidence that the split was leakage-free.
Section 2 of this notebook later measured that directly: a naive random split inflates
precision@50 by 26 points (0.90 vs. 0.64) versus the grouped split used here, so the grouped
design is now *measured*, not just assumed, to matter on this dataset."

I'm keeping the correction visible rather than quietly editing the Week 5 file, since the point
of this exercise is showing the audit trail, not a cleaner-looking history.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.